# Benchmark de Métodos de Busca com Vespa


Este notebook executa testes comparando diferentes métodos de recuperação de documentos usando o Vespa:
- **BM25**
- **Busca Semântica**
- **Busca Híbrida**
- **Fusão Linear**
Inclui medições de tempo usando `timing` do Vespa e métricas de avaliação.


In [1]:

import sys, os, json, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from vespa.application import Vespa
from tqdm.notebook import tqdm

from src.metrics import mrr_score, map_score, mr_score, mf1_score, mndcg_score

# Caminho dos módulos locais
src_path = os.path.abspath(os.path.join(os.getcwd(), "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from retrivers.vespa_retrievers import VespaBM25

model = SentenceTransformer("intfloat/e5-small-v2")
vespa = Vespa(url="http://localhost", port=8080)
vespa_retriever = VespaBM25()


/home/lunardonbruno/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def embed(text):
    return model.encode(text).tolist()


In [13]:
from sentence_transformers import SentenceTransformer
from vespa.application import Vespa
import pandas as pd

model = SentenceTransformer("intfloat/e5-small-v2")
app = Vespa(url="http://localhost", port=8080)

df = pd.read_pickle("/home/lunardonbruno/msmarco/subset_msmarco_train_0/subset_msmarco_train_0.01_9.pkl")

documents_df = pd.DataFrame([{"doc_id": d.doc_id, "content": d.text} for d in df["docs"].values()])
#get the first 30% documents_df = documents_df.head(30)
documents_df = documents_df.sample(frac=0.3, random_state=42)  # 30% dos dados
documents_df = documents_df.reset_index(drop=True)

# Embedding em lote (muito mais rápido)
contents = documents_df["content"].tolist()
embeddings = model.encode(contents, batch_size=32, show_progress_bar=True)

total_docs = len(documents_df)
progress_step = int(total_docs * 0.05)

print(f"Total de documentos: {total_docs}")

for idx, (row, emb) in enumerate(zip(documents_df.itertuples(), embeddings)):
    if idx % progress_step == 0:
        print(f"Progresso: {idx / total_docs:.2%}")

    response = app.feed_data_point(
        schema="msmarco",
        data_id=row.doc_id,
        fields={
            "id": row.doc_id,
            "content": row.content,
            "embedding": emb.tolist()
        }
    )
    if response.status_code != 200:
        print(f"Erro ao alimentar {row.doc_id}: {response.status_code}")

Batches: 100%|██████████| 261/261 [05:29<00:00,  1.26s/it]


Total de documentos: 8333
Progresso: 0.00%
Progresso: 4.99%
Progresso: 9.98%
Progresso: 14.98%
Progresso: 19.97%
Progresso: 24.96%
Progresso: 29.95%
Progresso: 34.95%
Progresso: 39.94%
Progresso: 44.93%
Progresso: 49.92%
Progresso: 54.91%
Progresso: 59.91%
Progresso: 64.90%
Progresso: 69.89%
Progresso: 74.88%
Progresso: 79.88%
Progresso: 84.87%
Progresso: 89.86%
Progresso: 94.85%
Progresso: 99.84%


In [14]:
queries_df = pd.DataFrame([{"query_id": q.query_id, "text": q.text} for q in df["queries"].values()])
qrels_df = pd.DataFrame([{"query_id": q.query_id, "doc_id": q.doc_id} for q in df["qrels"]])
relevant_dict = qrels_df.groupby("query_id")["doc_id"].apply(list).to_dict()


In [15]:
def query_vespa_semantic(query_text, hits=10):
    return vespa.query(body={
        "yql": f'select * from sources * where ([{{"targetNumHits":{hits}}}]nearestNeighbor(embedding, query_embedding));',
        "input.query(query_embedding)": embed(query_text),
        "ranking": "semantic",
        "hits": hits
    })

def query_vespa_hybrid(query_text, hits=10):
    return vespa.query(body={
        "yql": f'select * from sources * where userQuery() or ([{{"targetNumHits":{hits}}}]nearestNeighbor(embedding, query_embedding));',
        "query": query_text,
        "input.query(query_embedding)": embed(query_text),
        "ranking": "hybrid",
        "hits": hits
    })

def query_vespa_fusion(query_text, alpha=0.5, hits=10):
    return vespa.query(body={
        "yql": f'select * from sources * where userQuery() or ([{{"targetNumHits":{hits}}}]nearestNeighbor(embedding, query_embedding));',
        "query": query_text,
        "input.query(query_embedding)": embed(query_text),
        "input.query(alpha)": alpha,
        "ranking": "linear_fusion",
        "hits": hits
    })



In [23]:
class VespaBM25:
    def __init__(self):
        self.app = Vespa(url="http://localhost", port=8080)

    def run(self, query_text, k=100):
        response = self.app.query(body={
            "yql": "select * from sources * where userQuery();",
            "query": query_text,
            "ranking": "bm25",
            "hits": k
        })

        res_json = response.get_json()
        try:
            vespa_time = res_json["root"]["timing"]["total"]
        except KeyError:
            print("⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.")
            vespa_time = -1.0  # ou 0.0, se preferir

        hits = [(hit["fields"]["id"], hit.get("relevance", 1.0)) for hit in res_json.get("root", {}).get("children", [])]
        return hits, vespa_time


vespa_retriever = VespaBM25()

In [25]:

bm25_results, semantic_results, hybrid_results, fusion_results = {}, {}, {}, {}
bm25_times, semantic_times, hybrid_times, fusion_times = [], [], [], []

for _, row in queries_df.iterrows():
    qid, qtext = row["query_id"], row["text"]

    # BM25
    bm25_hits, bm25_time = vespa_retriever.run(query_text=qtext, k=10)
    bm25_results[qid] = bm25_hits
    bm25_times.append(bm25_time)

    # # Semantic
    # res = query_vespa_semantic(qtext)
    # sem_time = res.get_json().get("root", {}).get("timing", {}).get("total", -1)
    # semantic_times.append(sem_time)
    # semantic_results[qid] = [(hit["fields"]["id"], hit["relevance"]) for hit in res.hits]

    # # Hybrid
    # res = query_vespa_hybrid(qtext)
    # hyb_time = res.get_json().get("root", {}).get("timing", {}).get("total", -1)
    # hybrid_times.append(hyb_time)
    # hybrid_results[qid] = [(hit["fields"]["id"], hit["relevance"]) for hit in res.hits]

    # # Fusion
    # res = query_vespa_fusion(qtext, alpha=0.6)
    # fus_time = res.get_json().get("root", {}).get("timing", {}).get("total", -1)
    # fusion_times.append(fus_time) 
    # fusion_results[qid] = [(hit["fields"]["id"], hit["relevance"]) for hit in res.hits]


⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [VespaBM25.run] Warning: 'timing' not found in response JSON.
⚠️  [Vespa

In [26]:

def eval_all(results, name):
    print(f"### {name}")
    print("MRR:", mrr_score(results, relevant_dict, k=10))
    print("MAP:", map_score(results, relevant_dict, k=10))
    print("Recall:", mr_score(results, relevant_dict, k=10))
    print("F1:", mf1_score(results, relevant_dict, k=10))
    print("nDCG:", mndcg_score(results, relevant_dict, k=10))
    print()

eval_all(bm25_results, "BM25")
eval_all(semantic_results, "Semantic")
eval_all(hybrid_results, "Hybrid")
eval_all(fusion_results, "Linear Fusion (alpha=0.6)")


### BM25
MRR: 0.0
MAP: 0.0
Recall: 0.0
F1: 0.0
nDCG: 0.0

### Semantic
MRR: 0
MAP: 0
Recall: 0
F1: 0
nDCG: 0

### Hybrid
MRR: 0
MAP: 0
Recall: 0
F1: 0
nDCG: 0

### Linear Fusion (alpha=0.6)
MRR: 0
MAP: 0
Recall: 0
F1: 0
nDCG: 0

